# 7. Current noise from counting statistics

**Learning goals.** In this tutorial you will:

- select the lead channels included in counting statistics;
- read aggregate noise from `current_noise` and cross correlations from `current_noise_matrix`;
- construct a weighted spin-current noise from the covariance matrix;
- verify the counted first cumulant against the ordinary lead current;
- compare first-order Lindblad and fourth-order RTD counting; and
- identify the approximation and bandwidth checks required for RTD noise.

This tutorial assumes the interacting Anderson model from Tutorial 2, parameter sweeps from Tutorial 3, and the approximation-order discussion from Tutorial 6.

The tutorial was written by **Simon Wozny** and adapted from his [QmeQ noise example](https://github.com/si8881wo/qmeq-noise-example) at [source commit 0f81175](https://github.com/si8881wo/qmeq-noise-example/commit/0f81175c63b4f3846ac9c392c739615b038b9054). The original is copyright 2024 Simon Wozny and distributed under the BSD 2-Clause License; its notice is retained at `examples/licenses/qmeq-noise-example-BSD-2-Clause.txt`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import qmeq

print(qmeq.get_backend_status())

## The physical question

The stationary current is only the average of a stochastic sequence of tunnelling events. Counting statistics also asks how strongly the transferred particle number fluctuates. QmeQ returns the first two zero-frequency particle-current cumulants,

$$I=\lim_{t\to\infty}\frac{\langle\!\langle N(t)\rangle\!\rangle}{t},\qquad
S=\lim_{t\to\infty}\frac{\langle\!\langle N^2(t)\rangle\!\rangle}{t}.$$

We use a spin-degenerate Anderson dot with charging energy $U$. Four QmeQ lead channels represent the left and right reservoir for each spin. The right reservoir is hotter than the left, so the setup produces a thermoelectric current even at zero voltage bias.

**Prediction before calculating.** Hot electrons enter the level predominantly from the right and leave through the colder left side. With QmeQ's positive lead-to-dot convention, the aggregate currents should obey $I_R>0$, $I_L<0$, and $I_L+I_R=0$. The first entry of `current_noise` must equal the corresponding aggregate ordinary current.

All values use one arbitrary energy unit and $\hbar=k_\mathrm{B}=|e|=1$. Particle current and particle-number noise therefore both have units of inverse time, or energy in these units.

In [ ]:
nsingle = 2
level_energy = 500.0
charging_energy = 2000.0
hsingle = {(0, 0): level_energy, (1, 1): level_energy}
coulomb = {(0, 1, 1, 0): charging_energy}

temperature = 100.0
nleads = 4
mulst = {lead: 0.0 for lead in range(nleads)}
tlst = {0: temperature, 1: 2 * temperature,
        2: temperature, 3: 2 * temperature}

gamma_left, gamma_right = 1.5, 0.5
amplitude_left = np.sqrt(gamma_left / (2 * np.pi))
amplitude_right = np.sqrt(gamma_right / (2 * np.pi))
tleads = {(0, 0): amplitude_left, (1, 0): amplitude_right,
          (2, 1): amplitude_left, (3, 1): amplitude_right}

## Select the counted leads

QmeQ assigns an independent counting field to every entry of `countingleads`. `current_noise` retains the cumulants of their **aggregate** transfer, while `current_noise_matrix` contains the auto- and cross-correlations in exactly the order supplied. Here `(0, 2)` are the two left spin channels and `(1, 3)` the two right channels.

We count the cold left side with Lindblad and the hot right side with RTD. `Lindblad` uses whichever first-order backend is active. `RTDnoise` is a documented alias of the pure-Python counting implementation. RTD counting currently requires `off_diag_corrections=False`.

In [ ]:
common = dict(
    nsingle=nsingle, hsingle=hsingle, coulomb=coulomb, nleads=nleads,
    mulst=mulst, tlst=tlst, tleads=tleads, dband=1.0e7,
)

system_left = qmeq.Builder(
    **common, countingleads=(0, 2), kerntype="Lindblad",
)
system_right = qmeq.Builder(
    **common, countingleads=(1, 3), kerntype="RTDnoise",
    off_diag_corrections=False,
)

## Current and zero-frequency noise

`system.current` remains lead resolved. `system.current_noise` is `[I, S]` for the aggregate counted leads, and `system.current_noise_matrix` is their symmetric zero-frequency covariance matrix. The counting calculation is an additional observable: it does not replace or change the ordinary current calculation.

Three checks are available immediately: the stationary lead currents sum to zero, the first counted cumulant equals the sum over the selected ordinary lead currents, and the signs agree with the prediction above.

In [ ]:
system_left.solve()
system_right.solve()

left_ordinary = system_left.current[0] + system_left.current[2]
right_ordinary = system_right.current[1] + system_right.current[3]

print("Lead-resolved current, Lindblad:", system_left.current)
print("Lead-resolved current, RTD:", system_right.current)
print("Left aggregate [I, S], Lindblad:", system_left.current_noise)
print("Right aggregate [I, S], RTD:", system_right.current_noise.real)
print("Left noise covariance, Lindblad:\n",
      system_left.current_noise_matrix)
print("Right noise covariance, RTD:\n",
      system_right.current_noise_matrix.real)

assert np.isclose(sum(system_left.current), 0.0, atol=1e-12)
assert np.isclose(sum(system_right.current), 0.0, atol=1e-12)
assert np.isclose(system_left.current_noise[0], left_ordinary)
assert np.isclose(system_right.current_noise[0].real, right_ordinary)
assert np.isclose(system_left.current_noise[1],
                  system_left.current_noise_matrix.sum())
assert np.isclose(system_right.current_noise[1].real,
                  system_right.current_noise_matrix.sum().real)
assert left_ordinary < 0.0 < right_ordinary
print("\ncurrent conservation, counting identity, and signs verified")

The signs agree with the physical picture: particles enter from the hot right reservoir and leave through the cold left reservoir. The two aggregate currents above come from different approximations, so they are not expected to be exact negatives of one another. Current conservation applies separately within each solved system.

The noise $S$ is not available from `system.current` alone. It contains information about the temporal correlations between transfer events, which is precisely what the counting field adds.

## Cross correlations and weighted spin noise

For counted channels $i$ and $j$, `current_noise_matrix[i, j]` is the zero-frequency cross cumulant $S_{ij}$. Any real channel weights $w_i$ therefore define

$$I_w=\sum_i w_i I_i,\qquad S_w=\sum_{ij}w_iS_{ij}w_j.
$$

The left channels are ordered as spin up and spin down, so $w=(1/2,-1/2)$ gives the $S_z$ current and noise directly. The aggregate charge noise is the same quadratic form with $w=(1,1)$.

In [ ]:
left_channels = np.asarray(system_left.countingleads)
spin_weights = np.asarray([0.5, -0.5])
left_covariance = system_left.current_noise_matrix

spin_current = spin_weights @ system_left.current[left_channels]
spin_noise = spin_weights @ left_covariance @ spin_weights
charge_noise = np.ones(2) @ left_covariance @ np.ones(2)
three_run_identity = (
    2 * left_covariance[0, 0]
    + 2 * left_covariance[1, 1]
    - system_left.current_noise[1]
) / 4

print("left spin current:", spin_current)
print("left spin noise:", spin_noise)
print("left charge noise:", charge_noise)

np.testing.assert_allclose(left_covariance, left_covariance.T)
np.testing.assert_allclose(charge_noise, system_left.current_noise[1])
np.testing.assert_allclose(spin_noise, three_run_identity)
assert np.isclose(spin_current, 0.0, atol=1e-12)


## Gate sweep

A normal parameter sweep needs no special counting-statistics machinery: change the Hamiltonian, solve again, and record the two cumulants. We use the gate convention $\varepsilon=-V_g$.

For a direct visual comparison of transport direction, the plotted Lindblad current is $-I_L$ and the RTD current is $I_R$; both are positive when particles travel from the hot right side to the cold left side. At every gate value we independently require the counted first cumulant to equal the selected ordinary current.

In [ ]:
gate_values = np.linspace(-1500.0, 3500.0, 100)
ordinary_current = np.empty((2, len(gate_values)))
counted_current = np.empty_like(ordinary_current)
noise = np.empty_like(ordinary_current)

calculations = [
    (system_left, (0, 2), -1.0),  # plot -I_L
    (system_right, (1, 3), 1.0),  # plot  I_R
]
for approach_index, (system, counted_leads, direction) in enumerate(calculations):
    for gate_index, gate in enumerate(gate_values):
        system.change(hsingle={(0, 0): -gate, (1, 1): -gate})
        system.solve()
        aggregate = sum(system.current[lead] for lead in counted_leads)
        ordinary_current[approach_index, gate_index] = direction * aggregate
        counted_current[approach_index, gate_index] = (
            direction * system.current_noise[0].real
        )
        noise[approach_index, gate_index] = system.current_noise[1].real

np.testing.assert_allclose(counted_current, ordinary_current,
                           rtol=1e-10, atol=1e-12)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for index, (name, color) in enumerate([("Lindblad", "C0"),
                                        ("RTD", "C3")]):
    axes[0].plot(gate_values, ordinary_current[index], color=color, lw=3,
                 label=f"{name}, ordinary")
    axes[0].plot(gate_values, counted_current[index], color=color, ls="--",
                 label=f"{name}, counting")
    axes[1].plot(gate_values, noise[index], color=color, label=name)

axes[0].set(xlabel="$V_g$", ylabel="directed particle current",
            title="Mean current")
axes[1].set(xlabel="$V_g$", ylabel="$S$",
            title="Zero-frequency noise")
for axis in axes:
    axis.legend()
fig.tight_layout()

The dashed counting curves lie on top of the thick ordinary-current curves. That agreement is a consistency check on how the first cumulant was extracted; it is not a comparison between physical approximations. The separation between the blue Lindblad and red RTD curves is the approximation comparison. Noise provides an additional observable and need not track the mean current.

## Reading the three RTD result forms

RTD counting exposes the full result, its covariance matrix, and two order-resolved diagnostics:

- `current_noise` is `[I, S]` from the full fourth-order RTD kernel and stationary state;
- `current_noise_matrix` is the covariance matrix for the full result;
- `current_noise_first` and `current_noise_matrix_first` are the sequential result and covariance matrix; and
- `current_noise_o4trunc` is `[I_sequential, I_fourth_order, S_sequential, S_fourth_order]`, evaluated with a consistent fourth-order truncation.

Here *fourth order* means fourth order in the tunnelling Hamiltonian $H_T$, equivalently second order in the rate $\Gamma$. It does **not** mean fourth order in $\Gamma$.

In [ ]:
comparison_gate = 500.0
system_right.change(hsingle={(0, 0): -comparison_gate,
                             (1, 1): -comparison_gate})
system_right.solve()

print("full [I, S]:", np.asarray(system_right.current_noise).real)
print("full covariance:\n",
      np.asarray(system_right.current_noise_matrix).real)
print("sequential [I, S]:", np.asarray(system_right.current_noise_first).real)
print("sequential covariance:\n",
      np.asarray(system_right.current_noise_matrix_first).real)
print("consistent O(H_T^4) decomposition:",
      np.asarray(system_right.current_noise_o4trunc).real)

## Approximation and cutoff checks

The unequal-temperature RTD integrals retain `dband` as a finite wide-band regulator. The large value used here suppresses visible cutoff effects for this example, but one value cannot demonstrate convergence. For a production result, repeat the calculation with increasing `dband` and require **both** $I$ and $S$ to converge; noise generally needs the stricter check. `RTDBandwidthWarning` identifies clearly under-separated cutoffs, but the absence of a warning is not a convergence proof.

| requirement | why |
| --- | --- |
| `countingleads` is a nonempty ordered set of unique lead indices | fixes the row and column order of `current_noise_matrix` |
| counted $I$ equals the selected ordinary current | catches sign and lead-selection mistakes |
| RTD: `off_diag_corrections=False` | off-diagonal counting corrections are not implemented |
| RTD thermal bias: convergence in `dband` | unequal-temperature second-order integrals retain the cutoff |
| weak tunnelling, $\Gamma\ll T$ | Lindblad and RTD are perturbative approximations |
| no 2vN, electron-phonon, or matrix-free counting | these combinations deliberately raise `NotImplementedError` |

Only the first two zero-frequency **particle-current** cumulants are implemented. QmeQ does not yet provide arbitrary higher cumulants or energy-current noise. The auxiliary `current_noise_o4trunc` order decomposition has no matrix-valued companion. The [counting-statistics theory and API page](https://github.com/maiani/qmeq/blob/master/docs/docs/theory/counting-statistics.md) gives the kernel formulas and complete convention list.

## Exercises

1. Set all four temperatures equal. Predict the zero-bias mean current before solving, then check whether the equilibrium noise also vanishes.
2. Count all four lead channels at once. Use charge conservation to predict the quadratic form of `current_noise_matrix` with four unit weights.
3. Replace `spin_weights` by another real weight vector and verify the result against the corresponding matrix quadratic form.
4. Change `system_left.countingleads` from `(0, 2)` to `(1, 3)`, solve again, and verify the sign reversal of the first cumulant.
5. At `comparison_gate`, repeat the RTD calculation for successively larger `dband`. Compare the relative convergence of the current and the noise rather than assuming that one guarantees the other.